# Timeseries plotting
Visualises the BOLD timeseries for a single participant, showing positive, negative, and near-zero functional connectivity examples. Execute cells top to bottom. Only cells marked **[CONFIGURE]** require changes.

> Run `scripts/run_scripts.ipynb` before running this notebook.

## 1. Environment Setup **[OPTIONAL]**
Mounts Google Drive and installs dependencies when running on Colab. Skip if running locally.

> **NOTE: requires moving `PROJECT` folder to Google Drive.**

In [ ]:
# COLAB
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd '/content/drive/My Drive/PROJECT'

    !pip install -q numpy pandas matplotlib seaborn

    sys.path.insert(0, './product/src')

## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing.timeseries import preprocess_timeseries, preprocess_all
from data_io.save_load_dataset import load_dataset, load_timeseries
from utils.paths import get_project_root

root = get_project_root()

## 3. Configuration **[CONFIGURE]**

| Variable | Description |
|---|---|
| `PARTICIPANT_IDX` | Row index into metadata for the participant to visualise |

In [ ]:
PARTICIPANT_IDX = 0

## 4. Helper Functions

`get_roi_data` extracts the z-scored BOLD timeseries for both ROIs in a given feature. `plot_correlation_comparison` produces a three-panel diagnostic plot: Region A, Region B, and their overlap.

In [ ]:
def get_roi_data(feat_idx, participant_ts, feature_labels, full_data_row):
    """Extracts metadata and z-scored timeseries for a specific feature index."""
    info = feature_labels.iloc[int(feat_idx)]
    val  = full_data_row[int(feat_idx)]

    ts_i = participant_ts[:, int(info['roi_i'])]
    ts_j = participant_ts[:, int(info['roi_j'])]

    ts_i_z = (ts_i - np.mean(ts_i)) / np.std(ts_i)
    ts_j_z = (ts_j - np.mean(ts_j)) / np.std(ts_j)

    return info, val, ts_i_z, ts_j_z


def plot_correlation_comparison(pid, feat_idx, val, info, ts_i_z, ts_j_z, label_type=""):
    """Generates a 3-panel diagnostic plot for a single ROI pair."""
    fig, axes = plt.subplots(3, 1, figsize=(18, 10), dpi=300, sharex=True)

    fig.suptitle(
        f'Participant {pid} | {label_type} Correlation | Val: {val:.8f}\n'
        f'Feature: {info["desc_feature_name"]}',
        fontsize=16, y=0.98
    )

    # Top panel: Region A
    axes[0].plot(ts_i_z, color='#1f77b4', linewidth=1.5,
                 label=f"ROI {int(info['roi_i'])}")
    axes[0].set_title(
        f"Region A: {info['desc_i']} (ROI {int(info['roi_i'])})", loc='left')

    # Middle panel: Region B
    axes[1].plot(ts_j_z, color='#d62728', linewidth=1.5,
                 label=f"ROI {int(info['roi_j'])}")
    axes[1].set_title(
        f"Region B: {info['desc_j']} (ROI {int(info['roi_j'])})", loc='left')

    # Bottom panel: overlap
    axes[2].plot(ts_i_z, color='#1f77b4', linewidth=1.5, label=info['label_i'])
    axes[2].plot(ts_j_z, color='#d62728', linewidth=1.5, label=info['label_j'])
    axes[2].set_title(f'Overlap (Pearson $r$ = {val:.8f})', loc='left')
    axes[2].legend(loc='upper right', frameon=True)

    for ax in axes:
        ax.set_ylabel('BOLD (Z-score)')
        ax.grid(True, alpha=0.5)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[2].set_xlabel('Time (TR Volumes)')
    plt.show()

## 5. Load Dataset and Timeseries

Loads the ABIDE dataset and the raw timeseries for the selected participant. Fisher z-scores are converted to Pearson r for display.

In [ ]:
prefix = 'NYU_UCLA_PITT'
X, X_raw, y, metadata, feature_labels = load_dataset(prefix, verbose=False)

# Convert Fisher z to Pearson r for plotting
X_pearson = np.tanh(X_raw)

idx = metadata['X'].index[PARTICIPANT_IDX]
pid = metadata.loc[idx, 'ID']
X_idx = X[idx, :]

path        = root / f"product/data/raw/{prefix}/{pid}_rois_cc200.1D"
ts          = np.genfromtxt(path)
participant = preprocess_timeseries(ts)

print(f"Loaded timeseries for participant id: {pid}, idx={idx}:")
print(f"\n{participant}")
print(f"\n{metadata.iloc[idx]}")

## 6. Timeseries Plots

Plots three diagnostic examples for the selected participant: the feature with the highest positive correlation, the most negative correlation, and the feature closest to zero. Each plot shows Region A, Region B, and their overlaid timeseries.

In [ ]:
scenarios = {
    "Positive":  np.argmax(X_pearson[idx]),
    "Negative":  np.argmin(X_pearson[idx]),
    "Near-Zero": np.argmin(np.abs(X_pearson[idx])),
}

for label, f_idx in scenarios.items():
    info, val_r, tsi, tsj = get_roi_data(f_idx, participant, feature_labels, X_pearson[idx])
    plot_correlation_comparison(pid, f_idx, val_r, info, tsi, tsj, label)